# 06B OpenMM solvated system (Route A)

Route A (Amber family): polymer GAFF2 (ABCG2 charges, from `05A_amber_gaff2_parameterisation.ipynb`) · protein ff19SB · water OPC · engine OpenMM.

This notebook builds the solvated Route A system with AmberTools `tleap` and tests it briefly in OpenMM:

- **§1 Polymer in water:** the 05A polymer in an OPC box, neutralised, with salt.
- **§2 Polymer + protein (optional):** the same, plus a protein (ff19SB) from a PDB in which the polymer is already posed.
- **§3 OpenMM short test:** minimisation, 10 ps NVT and 10 ps NPT, with energies and density.
- **§4 Production files** for `07_hpc_workflows.ipynb` (Route A section).

For the dry sanity check of the polymer alone, use `06A_openmm_dry_polymer.ipynb`. Every step that runs `tleap` or OpenMM is behind a `RUN_…` flag that defaults to `False`.

## Caveat: GAFF2 with OPC

GAFF2 (and its ABCG2/AM1-BCC charges) was not parameterised against the OPC water model. GAFF2 + OPC (+ ff19SB for proteins) is a common combination, and ff19SB is recommended with OPC, but it has **not been validated for PHA**. Treat hydration-dependent results (solvation free energies, polymer–water interactions) with that in mind. The enzyme–PHA production runs used Route C (CHARMM36m + CGenFF + CHARMM TIP3P; notebooks 06C/06D), not this route.

## 1. Inputs

- `MOL2_PATH`, `FRCMOD_PATH`: the GAFF2 files from 05A (`<SYSTEM>.gaff2.mol2`, `<SYSTEM>.gaff2.frcmod`).
- `PROTEIN_PDB`: `None` for §1 (polymer in water). For §2, a protein PDB (ff19SB residue names, hydrogens optional) **in the same coordinate frame as the posed polymer mol2**, e.g. from docking.
- `OUTPUT_DIR`: a **new** folder for the solvated system.
- Box: `padding_nm` (default 1.2 nm between solute and box edge), `box_shape` (`"box"` or truncated octahedron `"oct"`), `salt_molar` (default 0.15 M Na⁺/Cl⁻ after neutralisation).

In [ ]:
from pathlib import Path
import sys

repo_root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src" / "iphasimulator").exists())
if str(repo_root / "src") not in sys.path:
    sys.path.insert(0, str(repo_root / "src"))
from iphasimulator.amber_solvation import (
    AmberSolvationSettings,
    OpenMMTestSettings,
    build_solvated_amber_system,
    run_openmm_short_test,
    write_openmm_production_files,
    write_tleap_input,
)

SYSTEM_NAME = "P3HB_4"
gaff2_dir = repo_root / "examples/output/md_tests" / SYSTEM_NAME / "gaff2"
MOL2_PATH = gaff2_dir / f"{SYSTEM_NAME}.gaff2.mol2"
FRCMOD_PATH = gaff2_dir / f"{SYSTEM_NAME}.gaff2.frcmod"
PROTEIN_PDB = None  # §2: Path to a protein PDB posed together with the polymer
OUTPUT_DIR = repo_root / "examples/output/md_tests" / SYSTEM_NAME / "openmm" / "solvated_system"

settings = AmberSolvationSettings(padding_nm=1.2, box_shape="box", salt_molar=0.15)
print("mol2 found:  ", MOL2_PATH.is_file(), MOL2_PATH)
print("frcmod found:", FRCMOD_PATH.is_file(), FRCMOD_PATH)
print("protein:     ", PROTEIN_PDB)
print("output:      ", OUTPUT_DIR, "(exists — choose a new folder)" if OUTPUT_DIR.exists() else "")

## §1 / §2 Build the solvated system with tleap

The build runs tleap twice in `OUTPUT_DIR`: first to count the waters for the chosen padding, then to neutralise, add `round(salt_molar × waters / 55.5)` ion pairs, and save `system.prmtop`, `system.inpcrd` and `system.pdb`. `tleap.in` and `tleap.log` are kept next to them. The ion parameters are the ones `leaprc.water.opc` loads (Li/Merz 12-6 set for OPC); the file name tleap actually loaded is printed.

With `PROTEIN_PDB` set, the input also sources `leaprc.protein.ff19SB` and combines protein and polymer before solvating (§2). The preview below shows the tleap input without running anything.

In [ ]:
import tempfile

with tempfile.TemporaryDirectory() as preview_dir:
    print(write_tleap_input(
        Path(preview_dir) / "tleap.in",
        mol2_name=MOL2_PATH.name,
        frcmod_name=FRCMOD_PATH.name,
        protein_pdb_name=Path(PROTEIN_PDB).name if PROTEIN_PDB else None,
        settings=settings,
        salt_pairs=0,
    ))

RUN_TLEAP = False

if not RUN_TLEAP:
    print("Set RUN_TLEAP = True to build the solvated system with tleap.")
else:
    system = build_solvated_amber_system(
        MOL2_PATH, FRCMOD_PATH, OUTPUT_DIR, protein_pdb_path=PROTEIN_PDB, settings=settings
    )
    print("Waters added:        ", system.water_count)
    print("Salt ion pairs added:", system.salt_ion_pairs)
    print("Ion parameter files: ", ", ".join(system.ion_parameter_files) or "none reported")
    print("prmtop:", system.prmtop_path)
    print("inpcrd:", system.inpcrd_path)

## §3 OpenMM short test

From `system.prmtop`: PME with a 1.0 nm cut-off, HBonds constraints, LangevinMiddle integrator at 300 K with a 2 fs step, and a Monte Carlo barostat at 1 bar (on only during NPT). The test minimises, then runs 10 ps NVT and 10 ps NPT, and reports potential energies and the final density. Water-dominated boxes should approach about 1.0 g/mL; 10 ps is far too short for equilibrium, so this only shows that the system is stable.

In [ ]:
RUN_OPENMM_TEST = False
openmm_settings = OpenMMTestSettings(nvt_ps=10, npt_ps=10, platform_name=None)  # e.g. "CUDA"

if not RUN_OPENMM_TEST:
    print("Set RUN_OPENMM_TEST = True to run the short OpenMM test.")
else:
    test = run_openmm_short_test(
        OUTPUT_DIR / "system.prmtop", OUTPUT_DIR / "system.inpcrd", OUTPUT_DIR / "short_test",
        settings=openmm_settings,
    )
    print(f"Minimised energy: {test.minimized_energy_kj_mol:.1f} kJ/mol")
    print(f"After NVT:        {test.nvt_energy_kj_mol:.1f} kJ/mol")
    print(f"After NPT:        {test.npt_energy_kj_mol:.1f} kJ/mol")
    print(f"Density:          {test.density_g_ml:.3f} g/mL")
    print("Report:", test.report_path)

## §4 Production files for HPC

Writes `run_openmm_production.py` (restartable from `production.chk`, same settings as §3) and `run_openmm_production.slurm` into `OUTPUT_DIR`. Nothing runs here; submit them from the Route A section of `07_hpc_workflows.ipynb`.

In [ ]:
WRITE_PRODUCTION_FILES = False
PRODUCTION_NS = 100
JOB_NAME = f"{SYSTEM_NAME}_OPC"

if not WRITE_PRODUCTION_FILES:
    print("Set WRITE_PRODUCTION_FILES = True after the system is built to write the HPC files.")
else:
    script, slurm = write_openmm_production_files(OUTPUT_DIR, production_ns=PRODUCTION_NS, job_name=JOB_NAME)
    print("Wrote", script)
    print("Wrote", slurm)